# Lab 23: The Smart Door (AI Decides, the ESP32 Moves)

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A smart door is a classic AIoT product. In this lab the **Jetson decides** who may come in and the **ESP32 moves the door** (a servo). The Jetson reads a **QR pass** with the camera (Lab 13), checks it against rules (who, which hours, how many wrong tries), and sends `servo` commands to the board (Lab 22). It also watches the **doorway with the detector** (Lab 7) so that the door never closes on a person. Every attempt is logged. You also think like a security engineer: how could someone **cheat** the door, and what can you do about it?

**By the end of this lab you can:**
1. Write access rules as a function and test it, including time windows.
2. Add a lockout after wrong tries and explain why.
3. Build a door controller with states (closed, open) and a timer.
4. Use AI for **safety**: keep the door open while a person is in the doorway.
5. Log every attempt and discuss the weaknesses of a QR pass.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Board.** Like Lab 22, the lab uses the **virtual ESP32** first. With the real board, use a small servo as the door (a card flap). Switch to the real board as in Part D of Lab 22: the Jetson program does not change.
> **Privacy and safety.** Use only the passes you make in this lab. A real door must always have a **safe way out** (a handle, a button): never trap a person.

## 2. Background

**Who does what.** The same split as Lab 22:

| Jetson (AI brain) | ESP32 (IoT body) |
|---|---|
| Reads the picture, finds the QR code, finds the people | Moves the servo when told |
| Decides: allow or deny | Confirms every command (`esp/ack`) |
| Keeps the rules, the timer, the log | Switches the alert off if the Jetson disappears |

**Access rules.** A pass is only a number. The rule decides what it is allowed to do. Our rule checks, in this order:
1. Is the code **known** (in `ROSTER`)?
2. Is it **inside the allowed hours** for the role (students 8:00 to 17:00, teachers always)?
3. Is the door **locked** after too many wrong tries?

**Lockout.** Somebody who tries many codes may guess one. After `MAX_FAILS` wrong tries in a row the door ignores **all** codes for `LOCKOUT` seconds. The wrong tries are also logged, so an attack can be seen later.

**The door as a state machine.** The door is always in one **state**: `CLOSED` or `OPEN`. Events change the state:

| State | Event | New state | Action |
|---|---|---|---|
| CLOSED | A valid pass | OPEN | Send `servo` to the open angle, start the timer |
| OPEN | The timer ends and the doorway is **empty** | CLOSED | Send `servo` to the closed angle |
| OPEN | A person is in the doorway | OPEN | Extend the timer |

**AI for safety.** A door that closes on a person is dangerous. The detector counts the persons whose box centre is inside the **doorway zone** (Lab 12). While one is there, the door stays open. This is the AI doing a safety job.

**Weaknesses of a QR pass.** A QR code is a picture. Anyone can **copy** it: a photo of a pass on a phone opens the door (a **replay attack**). Two people can enter on one pass (**tailgating**). Good systems add a second check, for example a code that changes every 30 seconds, or a person who must also be seen. In this lab you find the weaknesses and fix one.

**Fail-safe and fail-secure.** If the power or the network fails, should the door **open** (fail-safe, good for a fire exit) or **stay closed** (fail-secure, good for a store room)? There is no single right answer: it depends on the door. The servo of this lab simply stays where it is.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(23)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Plan and Prepare (30 min)

### Step 1: The Board and the Passes
The cell starts the board link (the virtual board, as in Lab 22) and makes the QR passes with `labview.qr_picture`. `ROSTER` says who has which code and role. Change the names to your team. The cell also makes a pass that is **not** in the roster (`qr_stranger.png`).

In [ ]:
# ---------------- Settings you can change ----------------
ROSTER = {                       # the text in the QR code: who and which role
    "DKA4-AMINAH": {"name": "Aminah", "role": "student"},
    "DKA4-AMIRUL": {"name": "Amirul", "role": "student"},
    "DKA4-TEACHER": {"name": "Teacher", "role": "teacher"},
}
ALLOWED_HOURS = {"student": (8, 17), "teacher": (0, 24)}     # allowed from this hour up to (not including) this hour
OPEN_SECONDS = 5.0               # how long the door stays open
MAX_FAILS = 3                    # wrong tries in a row before the lockout
LOCKOUT = 20.0                   # seconds the door ignores all passes after too many wrong tries
ATTEMPT_GAP = 3.0                # the same pass in view counts as one try per this many seconds
DOOR_CLOSED, DOOR_OPEN = 10, 100 # servo angles
DOORWAY = [(0.30, 0.15), (0.90, 0.15), (0.90, 0.97), (0.30, 0.97)]    # the doorway as (x, y) fractions of the picture
# ----------------------------------------------------------
import csv
import glob
import os
import shutil
import threading
import time
from datetime import datetime

import cv2
import numpy as np
from IPython.display import Image, display

import labview

board = labview.VirtualEsp32()
esp = labview.Esp32()
time.sleep(1)
print("Board online:", esp.online())


def make_qr(text, size=6):
    return labview.qr_picture(text, size)       # OpenCV, or the qrcode package when OpenCV is old


for text, who in list(ROSTER.items()) + [("DKA4-STRANGER", {"name": "stranger"})]:
    file = f"qr_{who['name'].lower()}.png"
    cv2.imwrite(file, make_qr(text))
    print(f"{file}: {text}")
display(Image(filename="qr_aminah.png", width=140))

Open the passes from the file browser on the left and show them on a phone later. If you have a printer, print them.

### Step 2: Think Like an Attacker
Before you build the door, list ways to cheat it.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Three ways to open the door without a valid pass:
- For each: what could the system do about it?:
- Is a door with a QR pass safe enough for a school store room? For a bank vault? Why?:


---
# Part B: The Access Rules (45 min)

### Step 1: The Rule Function
`check_access(code, when)` returns `(allowed, who, reason)`. It checks the code and the hours. The cell tests it with different codes and times.

In [ ]:
def check_access(code, when):
    """Allow or deny a pass. Returns (allowed, name, reason). `when` is a datetime."""
    person = ROSTER.get(code)
    if person is None:
        return False, "unknown", "unknown code"
    start, end = ALLOWED_HOURS.get(person["role"], (0, 0))
    if not (start <= when.hour < end):
        return False, person["name"], f"outside the allowed hours ({start}:00 to {end}:00)"
    return True, person["name"], "ok"


def at(hour):
    return datetime(2026, 11, 9, hour, 30)


for code, hour in (("DKA4-AMINAH", 10), ("DKA4-AMINAH", 20), ("DKA4-TEACHER", 3), ("DKA4-STRANGER", 10), ("", 10)):
    allowed, who, reason = check_access(code, at(hour))
    print(f"{code or '(empty)':<14} at {hour:>2}:30  ->  {'ALLOW' if allowed else 'DENY ':<5} {who:<8} {reason}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which tests were allowed and which denied?:
- Why is `unknown` the name for an unknown code? Why not show the code in the screen message?:
- Add a new role `guest` with the hours 9 to 12 in `ALLOWED_HOURS` and add a guest to `ROSTER`. Does the test work?:


### Step 2: Lockout after Wrong Tries
The class `AccessControl` adds the **try gap** (the same pass in view counts once per `ATTEMPT_GAP` seconds) and the **lockout**. The time is given as a parameter `now`, so the cell can test it quickly with made-up times.

In [ ]:
class AccessControl:
    def __init__(self):
        self.fails = 0
        self.locked_until = 0.0
        self.last_try = {}

    def attempt(self, code, now, when=None):
        """One pass shown to the camera. Returns None when it is the same pass as a moment ago, otherwise (allowed, name, reason)."""
        if now - self.last_try.get(code, -1e9) < ATTEMPT_GAP:
            return None                                          # still the same try
        self.last_try[code] = now
        if now < self.locked_until:
            return False, "-", f"locked for {self.locked_until - now:.0f} more seconds"
        allowed, name, reason = check_access(code, when or datetime.now())
        if allowed:
            self.fails = 0
        else:
            self.fails += 1
            if self.fails >= MAX_FAILS:
                self.locked_until = now + LOCKOUT
                self.fails = 0
                reason += f" | LOCKED for {LOCKOUT:.0f} s"
        return allowed, name, reason


control = AccessControl()
t = 1000.0
for step, code in enumerate(["DKA4-STRANGER", "DKA4-STRANGER", "DKA4-STRANGER", "DKA4-AMINAH", "DKA4-AMINAH", "DKA4-AMINAH"]):
    now = t + step * 4                                            # one pass every 4 seconds
    print(f"t = {step * 4:>2} s  {code:<14}", control.attempt(code, now, at(10)))
print("The same pass again after 1 second:", control.attempt("DKA4-AMINAH", t + 21, at(10)))
print("After the lockout (t = 50 s):      ", control.attempt("DKA4-AMINAH", t + 50, at(10)))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What happened after 3 wrong tries? Was the valid pass of Aminah accepted at t = 12 s?:
- Why does a lockout help against guessing? What is the cost for a real user?:
- What is the use of `ATTEMPT_GAP`?:


---
# Part C: The Door Controller and the Safety Check (45 min)

### Step 1: The Door State Machine
`Door` keeps the state and the timer and sends the servo commands. `update` is called for every picture with `blocked` (is a person in the doorway?). The cell tests it with fake times and a fake board link.

In [ ]:
class Door:
    def __init__(self, link):
        self.link = link
        self.state = "CLOSED"
        self.open_until = 0.0

    def open(self, now):
        """A valid pass: open the door and start (or restart) the timer."""
        if self.state == "CLOSED":
            self.link.send("servo", DOOR_OPEN)
            self.state = "OPEN"
        self.open_until = now + OPEN_SECONDS
        return "opened"

    def update(self, now, blocked):
        """Called for every picture. Returns "closed" when the door closed now, otherwise None."""
        if self.state != "OPEN":
            return None
        if blocked:
            self.open_until = max(self.open_until, now + 2.0)     # a person is in the doorway: keep the door open
            return None
        if now >= self.open_until:
            self.link.send("servo", DOOR_CLOSED)
            self.state = "CLOSED"
            return "closed"
        return None


class FakeLink:
    def __init__(self):
        self.sent = []

    def send(self, name, value):
        self.sent.append((name, value))


fake = FakeLink()
door = Door(fake)
door.open(100.0)
print("t=100 opened:", door.state, fake.sent)
print("t=103 blocked:", door.update(103.0, blocked=True), door.state, "(timer was 105, now extended)")
print("t=106 blocked:", door.update(106.0, blocked=True), door.state)
print("t=107 free:   ", door.update(107.0, blocked=False), door.state, "(timer 108)")
print("t=109 free:   ", door.update(109.0, blocked=False), door.state, fake.sent)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- When did the door close and why not earlier?:
- What does the door do when a second pass is shown while it is open (look at `open`)?:


### Step 2: Is a Person in the Doorway?
The detector finds the people. `person_in_doorway` returns `True` if the **centre** of any person box is inside the `DOORWAY` polygon. `cv2.pointPolygonTest` does the geometry (as in Lab 15). The cell checks it first.

In [ ]:
def doorway_polygon(shape):
    h, w = shape[:2]
    return np.array([[int(x * w), int(y * h)] for x, y in DOORWAY], dtype=np.int32)


def person_in_doorway(boxes, shape):
    """boxes: a list of (x1, y1, x2, y2) of the people. True when a box centre is inside the doorway."""
    zone = doorway_polygon(shape).reshape(-1, 1, 2)
    return any(cv2.pointPolygonTest(zone, ((x1 + x2) / 2, (y1 + y2) / 2), False) >= 0 for x1, y1, x2, y2 in boxes)


shape = (480, 640, 3)
print("A person in the doorway:      ", person_in_doorway([(300, 150, 420, 460)], shape), "(expected True)")
print("A person far left of the door:", person_in_doorway([(10, 150, 80, 460)], shape), "(expected False)")
print("Nobody:                       ", person_in_doorway([], shape), "(expected False)")

### Step 3: Load the Detector
The detector of Lab 8 counts the people (class 0). If the engine is missing, `yolo26n.pt` is used.

In [ ]:
import torch
from ultralytics import YOLO

MODEL = "yolo26n_fp16.engine"
for source in ["../lab8/yolo26n_fp16.engine", "../lab9/yolo26n_fp16.engine", "../lab8/yolo26n.pt", "../models/yolo26n.pt"]:
    name = os.path.basename(source)
    if os.path.exists(source) and not os.path.exists(name):
        shutil.copy(source, name)
if not os.path.exists(MODEL):
    print(f"{MODEL} not found, using yolo26n.pt instead (slower)")
    MODEL = "yolo26n.pt"
device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL, task="detect")
print("Using model:", MODEL)

---
# Part D: Run the Smart Door (35 min)

### Step 1: The Live Door
The program reads the QR codes (the Lab 13 detector), asks `AccessControl`, opens the `Door` for valid passes, keeps it open while a person is in the doorway and writes every try in `access_log.csv` and `access_log.xlsx`. It also publishes the door state to the topic `door/state` (retained), so a phone can show it. The picture shows the zone, the state, the last result and the lockout.

Show a pass (on a phone or on paper) to the camera. Try the valid passes, the stranger, and the same pass twice. Stand in the doorway after the timer ends. Press **Stop** when you are done.

In [ ]:
control, door = AccessControl(), Door(esp)
run = {"note": "Show a pass", "warned": False, "last_state": None}


def read_codes(frame):
    return labview.read_qr(frame)       # pyzbar reads the codes (the OpenCV of this Jetson cannot)


def write_excel():
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Access"
    with open("access_log.csv", newline="") as f:
        for row in csv.reader(f):
            sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCDEF", (20, 16, 12, 8, 46, 10)):
        sheet.column_dimensions[column].width = width
    book.save("access_log.xlsx")


def log_access(code, name, allowed, reason):
    first = not os.path.exists("access_log.csv")
    with open("access_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "code", "name", "result", "reason", "door"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), code, name, "ALLOW" if allowed else "DENY", reason, door.state])
    write_excel()


def process(frame, state):
    now = time.time()
    result = model(frame, conf=0.4, device=device, classes=[0], verbose=False)[0]
    people = [tuple(int(v) for v in box) for box in result.boxes.xyxy.tolist()]
    blocked = person_in_doorway(people, frame.shape)
    shown = frame.copy()
    for x1, y1, x2, y2 in people:
        cv2.rectangle(shown, (x1, y1), (x2, y2), (200, 200, 0), 2)

    for code, corners in read_codes(frame):
        outcome = control.attempt(code, now)
        if outcome is None:
            continue
        allowed, name, reason = outcome
        if allowed:
            door.open(now)
        log_access(code, name, allowed, reason)
        run["note"] = f"{'ALLOW' if allowed else 'DENY'} {name}: {reason}"
        cv2.polylines(shown, [corners.reshape(-1, 1, 2)], True, (0, 200, 0) if allowed else (0, 0, 255), 3)
    door.update(now, blocked)

    if door.state != run["last_state"]:
        run["last_state"] = door.state
        esp.client.publish("door/state", door.state, retain=True)        # for a phone app
    cv2.polylines(shown, [doorway_polygon(frame.shape)], True, (0, 255, 255) if blocked else (120, 120, 120), 2)
    colour = (0, 200, 0) if door.state == "OPEN" else (0, 0, 255)
    cv2.putText(shown, f"DOOR {door.state}" + (f" (closes in {max(0, door.open_until - now):.0f} s)" if door.state == "OPEN" else ""), (10, 34),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, colour, 3)
    if blocked:
        cv2.putText(shown, "PERSON IN THE DOORWAY", (10, 64), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    if now < control.locked_until:
        cv2.putText(shown, f"LOCKED {control.locked_until - now:.0f} s", (10, 92), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    cv2.putText(shown, run["note"][:70], (10, shown.shape[0] - 34), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    cv2.putText(shown, (board.describe() if run.get("virtual", True) else "") + ("" if esp.online() else "   BOARD OFFLINE"),
                (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    return shown


esp.send("servo", DOOR_CLOSED)
live = labview.live(process)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the door open for each valid pass and stay closed for the stranger?:
- What did the door do when you stood in the doorway after the timer? Did it close on you?:
- What happened after 3 wrong tries?:
- FPS shown and how long it took from showing the pass to the servo command:


Look at the log:

In [ ]:
if os.path.exists("access_log.csv"):
    with open("access_log.csv") as f:
        print(f.read())
else:
    print("No tries yet.")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many tries are in the log? How many allowed and denied?:
- What can the school learn from the log (for example who came late)?:


---
# Part E: Break It, Fix It and Present (25 min)

### Step 1: Attack Your Own Door
Test each attack and write what the door did. Then fix one weakness.

| Test | Attack | Expected | What happened |
|---|---|---|---|
| 1 | A valid pass | Door opens, one log line | |
| 2 | A photo of the pass of Aminah taken with another phone (a replay) | The door opens: this is the weakness | |
| 3 | The stranger pass 3 times | Lockout | |
| 4 | A valid pass during the lockout | Denied (locked) | |
| 5 | Two people walk in on one pass (tailgating) | Allowed, only one pass logged: the weakness | |
| 6 | Stand in the doorway when the timer ends | The door waits | |
| 7 | Stop the notebook while the door is open | The door stays open: fail-secure or fail-safe? | |
| 8 | The board is offline | `BOARD OFFLINE` shown; what does the door state say? | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4:
- Test 5 to 8:
- Which weakness is the most serious for a school store room?:


### Step 2: Fix One Weakness
Choose **at least one** and build it. Change the code in the cells above, run that cell again, then run the live cell again.

- **Replay:** a pass that **changes**. Add the minute to the code (`DKA4-AMINAH-` followed by `datetime.now().strftime("%H%M")`) and make a QR that you refresh every minute, or accept a code only if it is **new** (remember the used codes).
- **Second factor:** the door opens only if the pass is shown **and** a person is seen in the doorway zone (a pass in an empty picture is a photo, not a person).
- **Tailgating:** count the people in the doorway while the door is open. If more than one person walks in on one pass, publish an alert on `door/alert` and make the board **blink**.
- **A button inside:** `esp/cmd/...` for exit: a keyword from Lab 17 (`start`) opens the door from the inside.
- **Voice:** speak the result with Lab 18: "Welcome Aminah", "Access denied".
- **Phone:** show `door/state` on your phone with the IoT MQTT Panel app and send `open` from the phone (a remote button).

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Weakness you fixed and how:
- Which lines did you change?:
- Test the attack again: does the fix work? Is there a new weakness?:


### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: a valid pass, a stranger and the lockout, the doorway safety, the log, and the weakness you found and fixed.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your design (what is AI and what is IoT):
- Rules used: roles, hours, `MAX_FAILS`, `LOCKOUT`, `OPEN_SECONDS`:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- The weakness you fixed:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** In Lab 24 the board sends **sensor data** all day, and the Jetson uses AI to find unusual values and shows a live dashboard.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The QR passes are made and the stranger pass is denied | |
| 2 | `check_access` is tested with known, unknown and out-of-hours passes | |
| 3 | The lockout works and the try gap is explained | |
| 4 | The door state machine test is explained (open, extended, closed) | |
| 5 | The doorway check works with a real person | |
| 6 | The live door opens for a valid pass and moves the servo (virtual or real) | |
| 7 | The log is shown with allowed and denied tries | |
| 8 | At least one attack is tested and one weakness is fixed | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Which parts of the smart door are AI and which are IoT?:
- 2. List the checks of `check_access` in order. Why is the code checked first?:
- 3. What is a lockout and what attack does it stop?:
- 4. Draw (or describe) the two states of the door and the events that change them.:
- 5. How does the AI keep people safe at this door?:
- 6. What is a replay attack? Give one way to defend against it.:
- 7. What is tailgating?:
- 8. A fire exit door and a store room door: should each be fail-safe or fail-secure? Explain.:
- 9. Which is the strongest access system? (a) One QR code for all (b) A QR code that changes every 30 seconds plus a person seen at the door (c) A password on a sticky note (d) A door that is always open:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The pass is not read | Too small, shiny, tilted | 20 to 30 cm, flat, the whole code with its white border (Lab 13) |
| `unknown code` for a valid pass | The text in the QR is not exactly in `ROSTER` | Make the pass again with the cell of Part A. Check the capitals. |
| `outside the allowed hours` | The clock of the Jetson or the hours of the role | Check the time (`datetime.now()`), change `ALLOWED_HOURS` |
| The door does not open | The board is offline, or the pass is denied | Read the text on the picture and the log. Run `esp.online()`. |
| The door never closes | A person (or a picture of one) is in the doorway zone | Move out of the zone, or change `DOORWAY` |
| The door closes while a person is there | The detector misses the person | Lower the confidence in `process` (0.4), improve the light, check `DOORWAY` |
| Locked and cannot try again | The lockout is running | Wait `LOCKOUT` seconds, or lower `LOCKOUT` for tests |
| The same pass is logged many times | `ATTEMPT_GAP` is too short | Raise `ATTEMPT_GAP` |
| `Cannot open the camera` | Another program uses the camera | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| `AttributeError ... QRCodeEncoder` | The OpenCV of this Jetson is old | See Lab 13 (use the `qrcode` package) |
| `CUDA out of memory` | Other notebooks use the memory | Stop other live views, Kernel, Shut Down All Kernels, run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(23)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- OWASP, Authentication (ideas about passwords and lockout): https://cheatsheetseries.owasp.org/cheatsheets/Authentication_Cheat_Sheet.html
- Replay attack: https://en.wikipedia.org/wiki/Replay_attack
- TOTP, codes that change every 30 seconds: https://en.wikipedia.org/wiki/Time-based_one-time_password
- Finite-state machine: https://en.wikipedia.org/wiki/Finite-state_machine